In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files (they are expected in working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to compute summary for a dataframe with a timestamp column
def summarize_timeframe(df, ts_col='timestamp'):
    # Ensure timestamp numeric
    df[ts_col] = pd.to_numeric(df[ts_col], errors='coerce').astype('Int64')
    total = int(df.shape[0])
    if total == 0:
        return {
            'total_rows': 0,
            'min_ts_epoch': None,
            'min_ts_local': None,
            'max_ts_epoch': None,
            'max_ts_local': None
        }
    min_ts = int(df[ts_col].min())
    max_ts = int(df[ts_col].max())
    min_local = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz)
    max_local = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz)
    return {
        'total_rows': total,
        'min_ts_epoch': min_ts,
        'min_ts_local': min_local.strftime('%Y-%m-%d %H:%M:%S %Z'),
        'max_ts_epoch': max_ts,
        'max_ts_local': max_local.strftime('%Y-%m-%d %H:%M:%S %Z')
    }

# Metric summary
metric_time = summarize_timeframe(df_metric, 'timestamp')
metric_cmdb_unique = sorted(df_metric['cmdb_id'].dropna().unique().tolist())
# Top 20 KPI names
metric_top_kpis = df_metric['kpi_name'].value_counts().head(20).rename_axis('kpi_name').reset_index(name='count')

metric_summary = pd.DataFrame([{
    'total_rows': metric_time['total_rows'],
    'min_ts_epoch': metric_time['min_ts_epoch'],
    'min_ts_local': metric_time['min_ts_local'],
    'max_ts_epoch': metric_time['max_ts_epoch'],
    'max_ts_local': metric_time['max_ts_local'],
    'unique_cmdb_ids': ','.join(metric_cmdb_unique) if metric_cmdb_unique else ''
}])

# Trace summary
trace_time = summarize_timeframe(df_trace, 'timestamp')
trace_cmdb_unique = sorted(df_trace['cmdb_id'].dropna().unique().tolist())
trace_top = df_trace['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')

trace_summary = pd.DataFrame([{
    'total_rows': trace_time['total_rows'],
    'min_ts_epoch': trace_time['min_ts_epoch'],
    'min_ts_local': trace_time['min_ts_local'],
    'max_ts_epoch': trace_time['max_ts_epoch'],
    'max_ts_local': trace_time['max_ts_local'],
    'unique_cmdb_ids': ','.join(trace_cmdb_unique) if trace_cmdb_unique else ''
}])

# Log summary
log_time = summarize_timeframe(df_log, 'timestamp')
log_cmdb_unique = sorted(df_log['cmdb_id'].dropna().unique().tolist())
log_top = df_log['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')

log_summary = pd.DataFrame([{
    'total_rows': log_time['total_rows'],
    'min_ts_epoch': log_time['min_ts_epoch'],
    'min_ts_local': log_time['min_ts_local'],
    'max_ts_epoch': log_time['max_ts_epoch'],
    'max_ts_local': log_time['max_ts_local'],
    'unique_cmdb_ids': ','.join(log_cmdb_unique) if log_cmdb_unique else ''
}])

# Error logs: total rows and top 5 rows with selected columns
error_count = int(df_error.shape[0])
# Ensure timestamp column typed similarly for display
if 'timestamp' in df_error.columns:
    df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce').astype('Int64')
error_head = df_error.loc[:, ['timestamp','cmdb_id','message']].head(5)

# Final compact outputs (variables shown)
metric_summary, metric_top_kpis, trace_summary, trace_top, log_summary, log_top, error_count, error_head

```
Out[1]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing variables from previous analysis: metric_summary, metric_top_kpis, trace_summary, trace_top, log_summary, log_top, error_count, error_head

# Helper to get compact list of cmdb ids (first 10)
def compact_cmdb_list(cmdb_csv):
    if not cmdb_csv:
        return ""
    items = cmdb_csv.split(',')
    if len(items) > 10:
        return ','.join(items[:10]) + ',... (total {})'.format(len(items))
    return ','.join(items)

# Extract metric info
m = metric_summary.iloc[0]
metric_cmdb_compact = compact_cmdb_list(m['unique_cmdb_ids'])
metric_top5 = metric_top_kpis.head(5).apply(lambda r: f"{r['kpi_name']} ({r['count']})", axis=1).tolist()

# Extract trace info
t = trace_summary.iloc[0]
trace_cmdb_compact = compact_cmdb_list(t['unique_cmdb_ids'])
trace_top5 = trace_top.head(5).apply(lambda r: f"{r['trace_name']} ({r['count']})", axis=1).tolist()

# Extract log info
l = log_summary.iloc[0]
log_cmdb_compact = compact_cmdb_list(l['unique_cmdb_ids'])
log_top_items = log_top.head(5).apply(lambda r: f"{r['log_name']} ({r['count']})", axis=1).tolist()

# Error logs: compact display of top rows (convert timestamps to UTC+8)
err_rows = []
for _, row in error_head.iterrows():
    ts = int(row['timestamp']) if pd.notna(row['timestamp']) else None
    if ts is not None:
        local = pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        local = 'N/A'
    err_rows.append(f"{local} | {row.get('cmdb_id','')} | {str(row.get('message',''))[:200]}")

# Build summary string
summary_text = (
    "Summary of telemetry files (compact):\n\n"
    "metric.csv:\n"
    f" - total rows: {int(m['total_rows'])}\n"
    f" - time range (UTC+8): {m['min_ts_local']}  to  {m['max_ts_local']}\n"
    f" - unique cmdb_ids (sample/total): {metric_cmdb_compact}\n"
    f" - top KPIs (top 5): {', '.join(metric_top5)}\n\n"
    "trace.csv:\n"
    f" - total rows: {int(t['total_rows'])}\n"
    f" - time range (UTC+8): {t['min_ts_local']}  to  {t['max_ts_local']}\n"
    f" - unique cmdb_ids (sample/total): {trace_cmdb_compact}\n"
    f" - top traces (top 5): {', '.join(trace_top5)}\n\n"
    "log.csv:\n"
    f" - total rows: {int(l['total_rows'])}\n"
    f" - time range (UTC+8): {l['min_ts_local']}  to  {l['max_ts_local']}\n"
    f" - unique cmdb_ids (sample/total): {log_cmdb_compact}\n"
    f" - top logs (top items): {', '.join(log_top_items)}\n\n"
    "error_logs.csv:\n"
    f" - total rows: {int(error_count)}\n"
    f" - top rows (timestamp UTC+8 | cmdb_id | message snippet):\n"
    + ("\n".join("   - " + r for r in err_rows) if err_rows else "   - (no rows)")
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  min_ts_epoch             min_ts_local  max_ts_epoch             max_ts_local                                    unique_cmdb_ids
0       27659    1615086000  2021-03-07 11:00:00 CST    1615087800  2021-03-07 11:30:00 CST  IG01,IG02,MG01,MG02,Mysql01,Mysql02,Redis01,Re...,                                              kpi_name  count
0                  OSLinux-OSLinux_ZABBIX_Host_Uptime    365
1       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    365
2          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    365
3      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    360
4   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    359
5                             OSLinux-CPU_CPU_CPULoad    358
6                         OSLinux-CPU_CPU_CPUidleutil    358
7                              OSLinux-CPU_CPU_CPUWio    358
8                         OSLinux-CPU_CPU_CPUUserTime    356
9                          OSLinux-CPU_CPU_CPUSysTime    356
10       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    350
11                                             app.sr    341
12                                            app.cnt    341
13                                            app.mrt    341
14                                             app.rr    341
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    302
16                         OSLinux-CPU_CPU_CPUCpuUtil    296
17       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    291
18              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    284
19  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...    281,    total_rows  min_ts_epoch             min_ts_local  max_ts_epoch             max_ts_local                                    unique_cmdb_ids
0        6873    1615086000  2021-03-07 11:00:00 CST    1615087740  2021-03-07 11:29:00 CST  IG01,IG02,MG01,MG02,Tomcat01,Tomcat02,Tomcat03...,                        trace_name  count
0        trace.self.duration_mean    360
1         trace.self.duration_p95    360
2            trace.self.row_count    360
3      trace.to_MG01.duration_p95    240
4         trace.to_MG01.row_count    240
5     trace.to_MG02.duration_mean    240
6      trace.to_MG02.duration_p95    240
7         trace.to_MG02.row_count    240
8     trace.to_MG01.duration_mean    240
9       trace.from_MG02.row_count    120
10  trace.from_MG01.duration_mean    120
11   trace.from_MG01.duration_p95    120
12      trace.from_MG01.row_count    120
13   trace.from_MG02.duration_p95    120
14      trace.from_IG01.row_count    120
15  trace.from_IG02.duration_mean    120
16  trace.from_IG01.duration_mean    120
17   trace.from_IG01.duration_p95    120
18      trace.from_IG02.row_count    120
19   trace.from_IG02.duration_p95    120,    total_rows  min_ts_epoch             min_ts_local  max_ts_epoch             max_ts_local                                    unique_cmdb_ids
0         502    1615086000  2021-03-07 11:00:00 CST    1615087800  2021-03-07 11:30:00 CST  Mysql01,Tomcat01,Tomcat02,Tomcat03,Tomcat04,ap...,            log_name  count
0   log.error_count    203
1     log.row_count    203
2  log.gc_pause_max     96, 2,     timestamp   cmdb_id                                            message
0  1615087744  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
1  1615087749  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing df_metric if present; otherwise load from CSV
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure value is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Group by cmdb_id and kpi_name and compute required stats (using entire series)
agg = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    count = 'count',
    mean = 'mean',
    std = 'std',
    p95 = lambda x: x.quantile(0.95),
    p90 = lambda x: x.quantile(0.90),
    p50 = lambda x: x.quantile(0.50),
    p10 = lambda x: x.quantile(0.10),
    p5  = lambda x: x.quantile(0.05)
).reset_index()

# Clean and format columns: ensure count is int, round floats for compactness
agg['count'] = agg['count'].astype(int)
float_cols = ['mean','std','p95','p90','p50','p10','p5']
agg[float_cols] = agg[float_cols].round(6)

# Store full aggregated table in required kernel variable
metric_kpi_thresholds = agg.copy()

# Prepare compact top-20 table sorted by count descending
metric_kpi_thresholds_top20 = metric_kpi_thresholds.sort_values('count', ascending=False).loc[:, 
    ['cmdb_id','kpi_name','count','mean','std','p95','p90','p50','p10','p5']].head(20)

# Display the compact result (as per rules, show variable)
metric_kpi_thresholds_top20

```
Out[2]:
```
```python
# Summarize the aggregated metric_kpi_thresholds table as a plain-English string.
# Reuse metric_kpi_thresholds if available; otherwise rebuild it quickly.
import pandas as pd
import numpy as np

try:
    metric_kpi_thresholds
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    metric_kpi_thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        count = 'count',
        mean = 'mean',
        std = 'std',
        p95 = lambda x: x.quantile(0.95),
        p90 = lambda x: x.quantile(0.90),
        p50 = lambda x: x.quantile(0.50),
        p10 = lambda x: x.quantile(0.10),
        p5  = lambda x: x.quantile(0.05)
    ).reset_index()
    metric_kpi_thresholds['count'] = metric_kpi_thresholds['count'].astype(int)
    metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']] = metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']].round(6)

# Build compact summary
total_groups = int(metric_kpi_thresholds.shape[0])
top_groups = metric_kpi_thresholds.sort_values('count', ascending=False).head(10).loc[:, 
    ['cmdb_id','kpi_name','count','mean','std']]

lines = [
    f"Aggregated metric.csv by cmdb_id and kpi_name:",
    f" - total component-KPI groups: {total_groups}",
    f" - top groups by sample count (showing up to 10):"
]

for i, row in enumerate(top_groups.itertuples(index=False), start=1):
    cmdb_id, kpi_name, count, mean, std = row
    lines.append(f"   {i}. {cmdb_id} | {kpi_name} | count={int(count)} | mean={float(mean):.6g} | std={float(std):.6g}")

lines.append("")
lines.append("Note: Many top groups have the same maximum sample count (31). Several KPIs show zero std (constant values) while others have measurable variance.")
summary = "\n".join(lines)

# Display the plain-English summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count          mean            std           p95           p90           p50           p10            p5
1607  apache02                        OSLinux-CPU_CPU_CPUidleutil     31  9.934847e+01       0.183552  9.953251e+01  9.952640e+01  9.936030e+01  9.919830e+01  9.918982e+01
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.177419e+01       2.728976  7.550000e+01  7.500000e+01  7.200000e+01  6.800000e+01  6.700000e+01
16        IG01                        OSLinux-CPU_CPU_CPUidleutil     31  7.399460e+01       0.147827  7.419270e+01  7.415127e+01  7.398540e+01  7.386260e+01  7.381402e+01
12        IG01                            OSLinux-CPU_CPU_CPULoad     31  1.026774e+00       0.042535  1.100000e+00  1.070000e+00  1.010000e+00  1.000000e+00  1.000000e+00
13        IG01                         OSLinux-CPU_CPU_CPUSysTime     31  4.172250e-01       0.071555  5.157320e-01  4.907260e-01  4.366000e-01  3.285000e-01  3.202760e-01
14        IG01                        OSLinux-CPU_CPU_CPUUserTime     31  2.531410e+01       0.051356  2.536962e+01  2.536395e+01  2.532850e+01  2.526627e+01  2.524590e+01
15        IG01                             OSLinux-CPU_CPU_CPUWio     31  5.137100e-02       0.056602  1.788000e-01  1.330000e-01  2.080000e-02  1.250000e-02  1.248900e-02
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09       0.000000  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09
988    Redis02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00       0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
990    Redis02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00       0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
923    Redis01          redis-Redis_6379_Redis  (used_memory_rss)     31  1.199203e+07  217885.044908  1.232896e+07  1.229619e+07  1.195213e+07  1.173914e+07  1.170022e+07
924    Redis01                      redis-Redis_6379_redis server     31  1.000000e+00       0.000000  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
912    Redis01          redis-Redis_6379_Redis  (redis_git_dirty)     31  0.000000e+00       0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
987    Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00       0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1405  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02       0.000000  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02
1392  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00       0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1393  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00       0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1394  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00       0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1395  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00       0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1399  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00       0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric and metric_kpi_thresholds if available, otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    
# Ensure types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

try:
    metric_kpi_thresholds
except NameError:
    # compute if missing (shouldn't normally happen)
    metric_kpi_thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        count='count',
        mean='mean',
        std='std',
        p95=lambda x: x.quantile(0.95),
        p90=lambda x: x.quantile(0.90),
        p50=lambda x: x.quantile(0.50),
        p10=lambda x: x.quantile(0.10),
        p5=lambda x: x.quantile(0.05)
    ).reset_index()
    metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']] = metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']].round(6)

# Filter window (inclusive)
start_ts = 1615086000
end_ts = 1615087800
mask_window = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
df_window = df_metric.loc[mask_window].copy()

# Join p95 from precomputed thresholds (global p95)
p95_lookup = metric_kpi_thresholds.loc[:, ['cmdb_id','kpi_name','p95']]
df_window = df_window.merge(p95_lookup, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# Mark anomalies: value > p95 (strict)
df_window['is_anomaly'] = False
# only mark when both value and p95 are numeric
mask_valid = df_window['value'].notna() & df_window['p95'].notna()
df_window.loc[mask_valid, 'is_anomaly'] = df_window.loc[mask_valid, 'value'] > df_window.loc[mask_valid, 'p95']

# Store full anomaly rows table in kernel variable
metric_anomalies = df_window.loc[df_window['is_anomaly'], ['timestamp','cmdb_id','kpi_name','value','p95']].copy()

# Aggregate anomalies by cmdb_id and kpi_name
if metric_anomalies.shape[0] == 0:
    anomalies_summary = pd.DataFrame({
        'message': ['No anomalies found using p95 in the time window 1615086000-1615087800']
    })
else:
    agg = metric_anomalies.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count = ('value','count'),
        earliest_anomaly_ts = ('timestamp','min'),
        max_value = ('value','max'),
        p95 = ('p95','first')
    ).reset_index()
    # Convert earliest_anomaly_ts to UTC+8 readable string
    agg['earliest_anomaly_ts'] = agg['earliest_anomaly_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    # Round numeric columns for compactness
    agg['max_value'] = agg['max_value'].round(6)
    agg['p95'] = agg['p95'].round(6)
    # Sort by anomaly_count desc and keep top 20
    anomalies_summary = agg.sort_values('anomaly_count', ascending=False).loc[:, 
        ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','max_value','p95']].head(20)

# Display compact summary (and metric_anomalies is stored in kernel)
anomalies_summary

```
Out[3]:
```
```python
# Build a plain-English summary string based on existing anomalies_summary in kernel
try:
    anomalies_summary
except NameError:
    raise RuntimeError("anomalies_summary not found in kernel. Please run the previous analysis cell first.")

# If anomalies_summary contains a 'message' column, it indicates no anomalies were found.
if 'message' in anomalies_summary.columns and anomalies_summary.shape[0] == 1:
    summary = f"No anomalies found using p95 in the time window 1615086000-1615087800 (UTC+8)."
else:
    total_groups = int(anomalies_summary.shape[0])
    top_n = min(10, total_groups)
    lines = [
        f"Anomaly summary for window 1615086000–1615087800 (UTC+8):",
        f" - total anomalous component-KPI groups: {total_groups}",
        f" - Top {top_n} groups by anomaly_count:"
    ]
    # Ensure sorted by anomaly_count desc (it should be already)
    top_df = anomalies_summary.sort_values('anomaly_count', ascending=False).head(top_n)
    for i, row in enumerate(top_df.itertuples(index=False), start=1):
        cmdb_id = getattr(row, 'cmdb_id')
        kpi_name = getattr(row, 'kpi_name')
        anomaly_count = int(getattr(row, 'anomaly_count'))
        earliest = getattr(row, 'earliest_anomaly_ts')
        max_value = float(getattr(row, 'max_value'))
        p95 = float(getattr(row, 'p95'))
        lines.append(f"   {i}. {cmdb_id} | {kpi_name} | anomalies={anomaly_count} | earliest={earliest} | max_value={max_value:.6g} | p95={p95:.6g}")
    summary = "\n".join(lines)

# Display the plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         kpi_name  anomaly_count      earliest_anomaly_ts     max_value           p95
377       Redis02  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent              5  2021-03-07 11:02:00 CST  2.992940e+01  2.992940e+01
489      Tomcat02     OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc              4  2021-03-07 11:01:00 CST  8.051100e+01  8.051100e+01
85           MG01         JVM-Threads_7779_JVM_ThreadCount_Threads              2  2021-03-07 11:04:00 CST  7.700000e+01  7.650000e+01
405  ServiceTest3                                          app.mrt              2  2021-03-07 11:29:00 CST  6.000000e+01  2.486868e+01
406  ServiceTest4                                          app.cnt              2  2021-03-07 11:10:00 CST  2.320000e+02  2.160000e+02
407  ServiceTest4                                          app.mrt              2  2021-03-07 11:29:00 CST  6.000000e+01  2.539437e+01
408  ServiceTest5                                          app.cnt              2  2021-03-07 11:18:00 CST  4.270000e+02  3.775000e+02
409  ServiceTest5                                          app.mrt              2  2021-03-07 11:29:00 CST  6.000000e+01  2.139984e+01
606      apache01                      OSLinux-CPU_CPU_CPUidleutil              2  2021-03-07 11:01:00 CST  7.407269e+01  7.381797e+01
84           MG01                  JVM-Runtime_7779_JVM_JVM_Uptime              2  2021-03-07 11:29:00 CST  2.150859e+06  2.150790e+06
604      apache01                      OSLinux-CPU_CPU_CPUUserTime              2  2021-03-07 11:14:00 CST  7.246900e+00  7.224360e+00
603      apache01                       OSLinux-CPU_CPU_CPUSysTime              2  2021-03-07 11:04:00 CST  1.842730e+01  1.836931e+01
602      apache01                          OSLinux-CPU_CPU_CPULoad              2  2021-03-07 11:26:00 CST  2.010000e+00  1.935000e+00
410  ServiceTest6                                          app.cnt              2  2021-03-07 11:12:00 CST  2.680000e+02  2.520000e+02
411  ServiceTest6                                          app.mrt              2  2021-03-07 11:29:00 CST  6.000000e+01  2.433727e+01
412  ServiceTest7                                          app.cnt              2  2021-03-07 11:14:00 CST  2.920000e+02  2.560000e+02
605      apache01                           OSLinux-CPU_CPU_CPUWio              2  2021-03-07 11:00:00 CST  1.184870e+01  8.448950e+00
392       Redis02           redis-Redis_6379_Redis  (used_cpu_sys)              2  2021-03-07 11:28:00 CST  4.515771e+04  4.515535e+04
393       Redis02          redis-Redis_6379_Redis  (used_cpu_user)              2  2021-03-07 11:28:00 CST  1.648381e+04  1.648312e+04
4            IG01                  JVM-Runtime_7778_JVM_JVM_Uptime              2  2021-03-07 11:29:00 CST  1.668450e+05  1.667820e+05```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables if present, otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure correct dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Ensure metric_kpi_thresholds exists (global thresholds computed from entire series)
try:
    metric_kpi_thresholds
except NameError:
    metric_kpi_thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        count='count',
        mean='mean',
        std='std',
        p95=lambda x: x.quantile(0.95),
        p90=lambda x: x.quantile(0.90),
        p50=lambda x: x.quantile(0.50),
        p10=lambda x: x.quantile(0.10),
        p5=lambda x: x.quantile(0.05)
    ).reset_index()
    metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']] = metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']].round(6)

# Define incident window
start_ts = 1615086000
end_ts = 1615087800

# Filter metric rows to the incident window
mask_window = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
df_window = df_metric.loc[mask_window].copy()

# Join p95 threshold (global)
p95_lookup = metric_kpi_thresholds.loc[:, ['cmdb_id','kpi_name','p95']]
df_window = df_window.merge(p95_lookup, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# Mark anomalies where value > p95 (strict), only when both numeric
df_window['is_anomaly'] = False
mask_valid = df_window['value'].notna() & df_window['p95'].notna()
df_window.loc[mask_valid, 'is_anomaly'] = df_window.loc[mask_valid, 'value'] > df_window.loc[mask_valid, 'p95']

# Aggregate anomalies to get counts per group
anomaly_groups = df_window.loc[df_window['is_anomaly']].groupby(['cmdb_id','kpi_name'])['timestamp'].agg(
    anomaly_count='count'
).reset_index()

# If no anomalies, create empty runs table and return message
if anomaly_groups.shape[0] == 0:
    metric_fault_runs = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','length','max_value','p95','severity_ratio'])
    metric_fault_runs_top20 = metric_fault_runs.copy()
else:
    # Select top 10 groups by anomaly_count
    top_groups = anomaly_groups.sort_values('anomaly_count', ascending=False).head(10)
    runs = []
    for _, grp in top_groups.iterrows():
        cmdb = grp['cmdb_id']
        kpi = grp['kpi_name']
        # Get group's rows in window, sorted by timestamp
        df_grp = df_window[(df_window['cmdb_id'] == cmdb) & (df_window['kpi_name'] == kpi)].sort_values('timestamp')
        # Keep only anomaly rows (value > p95)
        df_anom = df_grp.loc[df_grp['is_anomaly'], ['timestamp','value','p95']].copy()
        if df_anom.shape[0] == 0:
            continue
        # Identify consecutive runs where diff == 60 seconds
        df_anom['timestamp'] = df_anom['timestamp'].astype(int)
        df_anom['diff'] = df_anom['timestamp'].diff().fillna(9999).astype(int)
        # Start new run when diff != 60
        df_anom['new_run'] = (df_anom['diff'] != 60).astype(int)
        df_anom['run_id'] = df_anom['new_run'].cumsum()
        # Aggregate runs
        grp_runs = df_anom.groupby('run_id').agg(
            run_start_ts=('timestamp','min'),
            run_end_ts=('timestamp','max'),
            length=('timestamp','count'),
            max_value=('value','max'),
            p95=('p95','first')
        ).reset_index(drop=True)
        # Compute severity_ratio safely (if p95==0 set to NaN)
        grp_runs['severity_ratio'] = grp_runs.apply(
            lambda r: (r['max_value'] - r['p95'])/r['p95'] if pd.notna(r['p95']) and r['p95'] != 0 else np.nan,
            axis=1
        )
        # Attach cmdb and kpi
        grp_runs['cmdb_id'] = cmdb
        grp_runs['kpi_name'] = kpi
        # Reorder columns and append
        grp_runs = grp_runs.loc[:, ['cmdb_id','kpi_name','run_start_ts','run_end_ts','length','max_value','p95','severity_ratio']]
        runs.append(grp_runs)
    # Concatenate all runs
    if runs:
        metric_fault_runs = pd.concat(runs, ignore_index=True)
    else:
        metric_fault_runs = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','length','max_value','p95','severity_ratio'])
    # Round numeric columns for compactness
    if not metric_fault_runs.empty:
        metric_fault_runs['max_value'] = metric_fault_runs['max_value'].round(6)
        metric_fault_runs['p95'] = metric_fault_runs['p95'].round(6)
        metric_fault_runs['severity_ratio'] = metric_fault_runs['severity_ratio'].round(6)

    # Sort by severity_ratio descending and keep top 20 for display
    metric_fault_runs_top20 = metric_fault_runs.sort_values('severity_ratio', ascending=False).head(20)

# Store full runs table in kernel variable metric_fault_runs (already set)
# Display compact top-20 table as output variable
metric_fault_runs_top20

```
Out[4]:
```
Summary of detected anomaly runs (incident window 2021-03-07 11:00–11:30 UTC+8):

- Total reported runs (top 10 groups scanned): 19 runs. Most runs are short (length 1 or 2 points), i.e., brief spikes or two-minute consecutive spikes.

- Highest-severity runs (by severity_ratio):
  1. ServiceTest5 — app.mrt: run 2021-03-07 11:29–11:30 (UTC+8), length=2, max_value=60, p95=21.399984, severity_ratio≈1.8038 (≈180% above p95).
  2. ServiceTest3 — app.mrt: run 2021-03-07 11:29–11:30 (UTC+8), length=2, max_value=60, p95=24.86868, severity_ratio≈1.4127.
  3. ServiceTest4 — app.mrt: run 2021-03-07 11:29–11:30 (UTC+8), length=2, max_value=60, p95=25.39437, severity_ratio≈1.3627.

- Other notable runs:
  - ServiceTest5 — app.cnt had a 2-point run at 11:18–11:19 with max=427, p95=377.5, severity_ratio≈0.131.
  - Several groups (Redis filesystem inode percent, Tomcat memory NoCacheMemPerc, some Apache CPU metrics) show single-point anomalies or runs where max == p95 (severity_ratio = 0), i.e., threshold-equal values or trivial exceedances.

- Interpretation / next steps:
  - The most significant anomalies are in application-level KPIs (app.mrt) for ServiceTest3/4/5 around 11:29–11:30, with the largest relative exceedance on ServiceTest5. These should be prioritized for investigation.
  - Many other anomalies are single-minute spikes or threshold-equal values and appear less severe.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         kpi_name  run_start_ts  run_end_ts  length     max_value           p95  severity_ratio
15  ServiceTest5                                          app.mrt    1615087740  1615087800       2  6.000000e+01  2.139984e+01        1.803759
10  ServiceTest3                                          app.mrt    1615087740  1615087800       2  6.000000e+01  2.486868e+01        1.412673
13  ServiceTest4                                          app.mrt    1615087740  1615087800       2  6.000000e+01  2.539437e+01        1.362729
14  ServiceTest5                                          app.cnt    1615087080  1615087140       2  4.270000e+02  3.775000e+02        0.131126
12  ServiceTest4                                          app.cnt    1615087200  1615087200       1  2.320000e+02  2.160000e+02        0.074074
11  ServiceTest4                                          app.cnt    1615086600  1615086600       1  2.180000e+02  2.160000e+02        0.009259
9           MG01         JVM-Threads_7779_JVM_ThreadCount_Threads    1615087140  1615087140       1  7.700000e+01  7.650000e+01        0.006536
8           MG01         JVM-Threads_7779_JVM_ThreadCount_Threads    1615086240  1615086240       1  7.700000e+01  7.650000e+01        0.006536
17      apache01                      OSLinux-CPU_CPU_CPUidleutil    1615086420  1615086420       1  7.407269e+01  7.381797e+01        0.003451
16      apache01                      OSLinux-CPU_CPU_CPUidleutil    1615086060  1615086060       1  7.398714e+01  7.381797e+01        0.002292
18          MG01                  JVM-Runtime_7779_JVM_JVM_Uptime    1615087740  1615087800       2  2.150859e+06  2.150790e+06        0.000032
2        Redis02  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent    1615086840  1615086840       1  2.992940e+01  2.992940e+01        0.000000
1        Redis02  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent    1615086480  1615086480       1  2.992940e+01  2.992940e+01        0.000000
0        Redis02  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent    1615086120  1615086120       1  2.992940e+01  2.992940e+01        0.000000
7       Tomcat02     OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    1615086480  1615086480       1  8.051100e+01  8.051100e+01        0.000000
3        Redis02  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent    1615087260  1615087260       1  2.992940e+01  2.992940e+01        0.000000
4        Redis02  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent    1615087620  1615087620       1  2.992940e+01  2.992940e+01        0.000000
5       Tomcat02     OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    1615086060  1615086120       2  8.051100e+01  8.051100e+01        0.000000
6       Tomcat02     OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    1615086240  1615086240       1  8.051100e+01  8.051100e+01        0.000000```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing df_trace and df_error if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')

# Ensure types
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Compute global thresholds for each (cmdb_id, trace_name)
trace_thresholds = df_trace.groupby(['cmdb_id','trace_name'])['value'].agg(
    count='count',
    mean='mean',
    std='std',
    p95=lambda x: x.quantile(0.95),
    p90=lambda x: x.quantile(0.90),
    p50=lambda x: x.quantile(0.50),
    p10=lambda x: x.quantile(0.10),
    p5 =lambda x: x.quantile(0.05)
).reset_index()

# Round numeric columns for compact storage
num_cols = ['mean','std','p95','p90','p50','p10','p5']
trace_thresholds[num_cols] = trace_thresholds[num_cols].round(6)
trace_thresholds['count'] = trace_thresholds['count'].astype(int)

# Store in kernel variable as requested
trace_thresholds = trace_thresholds  # variable retained

# 2) Filter trace rows to incident window
start_ts = 1615086000
end_ts = 1615087800
mask_window = (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)
df_trace_window = df_trace.loc[mask_window].copy()

# Join p95 and p5 for anomaly decision
trace_thresh_lookup = trace_thresholds.loc[:, ['cmdb_id','trace_name','p95','p5']]
df_trace_window = df_trace_window.merge(trace_thresh_lookup, on=['cmdb_id','trace_name'], how='left', validate='m:1')

# 3) Flag anomalies:
# - duration metrics (trace_name contains 'duration' case-insensitive): anomaly if value > p95
# - row_count metrics (trace_name contains 'row_count'): anomaly if value < p5
def detect_anomaly(row):
    name = str(row['trace_name']).lower()
    val = row['value']
    p95 = row['p95']
    p5 = row['p5']
    is_anom = False
    reason = None
    if 'duration' in name:
        if pd.notna(val) and pd.notna(p95) and val > p95:
            is_anom = True
            reason = 'duration_high'
    if 'row_count' in name:
        if pd.notna(val) and pd.notna(p5) and val < p5:
            is_anom = True
            reason = 'rowcount_low'
    return pd.Series([is_anom, reason])

df_trace_window[['is_anomaly','anomaly_reason']] = df_trace_window.apply(detect_anomaly, axis=1)

# 4) Store full anomaly rows table in kernel variable trace_anomalies
trace_anomalies = df_trace_window.loc[df_trace_window['is_anomaly'], 
                                      ['timestamp','cmdb_id','trace_name','value','p95','p5','anomaly_reason']].copy()

# 5) Aggregate anomalies by cmdb_id and trace_name
if trace_anomalies.shape[0] == 0:
    trace_anomalies_summary = pd.DataFrame({
        'message': ['No trace anomalies detected in the window using defined rules']
    })
else:
    agg = trace_anomalies.groupby(['cmdb_id','trace_name']).agg(
        anomaly_count = ('value','count'),
        earliest_anomaly_ts = ('timestamp','min'),
        latest_anomaly_ts = ('timestamp','max'),
        max_value = ('value','max'),
        min_value = ('value','min'),
        p95 = ('p95','first'),
        p5 = ('p5','first')
    ).reset_index()
    # Convert earliest/latest to UTC+8 readable strings
    agg['earliest_anomaly_ts'] = agg['earliest_anomaly_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    agg['latest_anomaly_ts'] = agg['latest_anomaly_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    # Round numeric cols
    agg['max_value'] = agg['max_value'].round(6)
    agg['min_value'] = agg['min_value'].round(6)
    agg['p95'] = agg['p95'].round(6)
    agg['p5'] = agg['p5'].round(6)
    # Sort by anomaly_count desc then earliest_anomaly_ts
    trace_anomalies_summary = agg.sort_values(['anomaly_count','earliest_anomaly_ts'], ascending=[False, True]).loc[
        :, ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value','min_value','p95','p5']
    ].head(20)

# Store full anomalies table in kernel variable
trace_anomalies = trace_anomalies  # full rows stored

# 6) Error logs in window: up to 10 rows, timestamp converted to UTC+8 readable, message truncated to 200 chars
df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce').astype('Int64')
mask_err_window = (df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts)
error_head = df_error.loc[mask_err_window, ['timestamp','cmdb_id','message']].copy().head(10)
# convert timestamp to readable UTC+8
if not error_head.empty:
    error_head['timestamp'] = error_head['timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    error_head['message'] = error_head['message'].astype(str).str.slice(0,200)

# Final compact outputs: trace_thresholds (global stats stored), trace_anomalies_summary (compact), and error_head (compact)
trace_thresholds, trace_anomalies_summary, error_head

```
Out[5]:
```
```python
# Build a plain-English summary string based on existing kernel variables:
# trace_thresholds, trace_anomalies_summary, error_head
try:
    trace_thresholds
    trace_anomalies_summary
    error_head
except NameError:
    raise RuntimeError("Required variables (trace_thresholds, trace_anomalies_summary, error_head) not found in kernel.")

lines = []
lines.append("Trace thresholds and anomalies summary (incident window 1615086000–1615087800, UTC+8):")

# Global thresholds
total_trace_groups = int(trace_thresholds.shape[0])
lines.append(f" - Trace groups with global thresholds computed: {total_trace_groups}")

# Anomalies summary
if 'message' in trace_anomalies_summary.columns and trace_anomalies_summary.shape[0] == 1:
    lines.append(" - No trace anomalies detected in the window using the defined rules.")
else:
    total_anom_groups = int(trace_anomalies_summary.shape[0])
    lines.append(f" - Anomalous trace groups detected: {total_anom_groups} (showing top up to 5):")
    top_n = trace_anomalies_summary.head(5)
    for i, r in enumerate(top_n.itertuples(index=False), start=1):
        cmdb = getattr(r, 'cmdb_id')
        tname = getattr(r, 'trace_name')
        cnt = int(getattr(r, 'anomaly_count'))
        earliest = getattr(r, 'earliest_anomaly_ts')
        latest = getattr(r, 'latest_anomaly_ts')
        vmax = float(getattr(r, 'max_value'))
        vmin = float(getattr(r, 'min_value'))
        p95 = float(getattr(r, 'p95')) if pd.notna(getattr(r, 'p95')) else None
        p5 = float(getattr(r, 'p5')) if pd.notna(getattr(r, 'p5')) else None
        reason_hint = []
        lname = tname.lower()
        if 'duration' in lname:
            reason_hint.append('duration_high (value > p95)')
        if 'row_count' in lname:
            reason_hint.append('rowcount_low (value < p5)')
        hint = '; '.join(reason_hint) if reason_hint else 'anomaly'
        lines.append(f"   {i}. {cmdb} | {tname} | anomalies={cnt} | earliest={earliest} | latest={latest} | max={vmax:.6g} | min={vmin:.6g} | p95={p95} | p5={p5} | {hint}")

# Error logs in window
err_count = int(error_head.shape[0])
lines.append(f" - Error logs in window returned (sample up to 10): {err_count} rows")
if err_count > 0:
    lines.append("   Sample error log rows (timestamp UTC+8 | cmdb_id | message snippet):")
    for r in error_head.itertuples(index=False):
        ts = getattr(r, 'timestamp')
        cmdb = getattr(r, 'cmdb_id')
        msg = getattr(r, 'message')
        lines.append(f"    - {ts} | {cmdb} | {msg[:200]}")

# Final guidance
lines.append("Recommendation: Prioritize trace groups with multiple anomalies (higher anomaly_count) and duration_high reasons for investigation; review related error logs shown above.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                     trace_name  count        mean        std         p95         p90         p50         p10         p5
0        IG01  trace.from_root.duration_mean      1    0.530000        NaN    0.530000    0.530000    0.530000    0.530000   0.530000
1        IG01   trace.from_root.duration_p95      1    1.023550        NaN    1.023550    1.023550    1.023550    1.023550   1.023550
2        IG01      trace.from_root.row_count      1    4.000000        NaN    4.000000    4.000000    4.000000    4.000000   4.000000
3        IG01       trace.self.duration_mean     30    2.089484   8.438451    0.733363    0.699494    0.561368    0.410131   0.386362
4        IG01        trace.self.duration_p95     30    4.410240  10.510571    3.166732    3.143175    2.156500    2.066365   2.049525
..        ...                            ...    ...         ...        ...         ...         ...         ...         ...        ...
256  dockerB2     trace.to_MG01.duration_p95     30    0.216975   0.385608    1.012000    1.011100    0.019025    0.009480   0.009135
257  dockerB2        trace.to_MG01.row_count     30  187.633333  62.020844  256.100000  244.200000  206.500000   88.700000  72.800000
258  dockerB2    trace.to_MG02.duration_mean     30    0.056070   0.022492    0.090809    0.084637    0.053308    0.033676   0.016049
259  dockerB2     trace.to_MG02.duration_p95     30    0.377003   0.467319    1.013438    1.012075    0.038350    0.009980   0.008000
260  dockerB2        trace.to_MG02.row_count     30  194.000000  66.246145  292.500000  257.300000  210.000000  111.800000  71.250000

[261 rows x 10 columns],       cmdb_id                         trace_name  anomaly_count      earliest_anomaly_ts        latest_anomaly_ts    max_value  min_value          p95           p5
50       MG01      trace.from_dockerB1.row_count              2  2021-03-07 11:00:00 CST  2021-03-07 11:29:00 CST    81.000000     16.000   255.850000    81.450000
52       MG01      trace.from_dockerB2.row_count              2  2021-03-07 11:00:00 CST  2021-03-07 11:29:00 CST    62.000000     17.000   256.100000    72.800000
88       MG02      trace.from_dockerB1.row_count              2  2021-03-07 11:00:00 CST  2021-03-07 11:29:00 CST    68.000000     15.000   255.400000    71.150000
204  dockerB1               trace.self.row_count              2  2021-03-07 11:00:00 CST  2021-03-07 11:29:00 CST  1593.000000    235.000  3918.250000  1641.600000
207  dockerB1            trace.to_MG01.row_count              2  2021-03-07 11:00:00 CST  2021-03-07 11:29:00 CST    81.000000     16.000   255.850000    81.450000
210  dockerB1            trace.to_MG02.row_count              2  2021-03-07 11:00:00 CST  2021-03-07 11:29:00 CST    68.000000     15.000   255.400000    71.150000
218  dockerB2               trace.self.row_count              2  2021-03-07 11:00:00 CST  2021-03-07 11:29:00 CST  1568.000000    164.000  4180.600000  1611.650000
220  dockerB2            trace.to_MG01.row_count              2  2021-03-07 11:00:00 CST  2021-03-07 11:29:00 CST    62.000000     17.000   256.100000    72.800000
67       MG01        trace.to_dockerB2.row_count              2  2021-03-07 11:01:00 CST  2021-03-07 11:29:00 CST   149.000000      8.000   268.100000   156.200000
91       MG02      trace.from_dockerB2.row_count              2  2021-03-07 11:01:00 CST  2021-03-07 11:29:00 CST    69.000000      7.000   292.500000    71.250000
103      MG02        trace.to_dockerB1.row_count              2  2021-03-07 11:01:00 CST  2021-03-07 11:29:00 CST   140.000000     14.000   284.750000   148.550000
202  dockerB1          trace.from_MG02.row_count              2  2021-03-07 11:01:00 CST  2021-03-07 11:29:00 CST   140.000000     14.000   284.750000   148.550000
213  dockerB2          trace.from_MG01.row_count              2  2021-03-07 11:01:00 CST  2021-03-07 11:29:00 CST   149.000000      8.000   268.100000   156.200000
223  dockerB2            trace.to_MG02.row_count              2  2021-03-07 11:01:00 CST  2021-03-07 11:29:00 CST    69.000000      7.000   292.500000    71.250000
47       MG01      trace.from_dockerA2.row_count              2  2021-03-07 11:03:00 CST  2021-03-07 11:29:00 CST     6.000000      6.000    67.000000     7.350000
193  dockerA2            trace.to_MG01.row_count              2  2021-03-07 11:03:00 CST  2021-03-07 11:29:00 CST     6.000000      6.000    67.000000     7.350000
44       MG01      trace.from_dockerA1.row_count              2  2021-03-07 11:04:00 CST  2021-03-07 11:29:00 CST     4.000000      1.000    65.950000     5.800000
45       MG01  trace.from_dockerA2.duration_mean              2  2021-03-07 11:04:00 CST  2021-03-07 11:10:00 CST     0.248778      0.145     0.135313     0.006746
175  dockerA1               trace.self.row_count              2  2021-03-07 11:04:00 CST  2021-03-07 11:29:00 CST  1455.000000     56.000  2747.300000  1469.850000
178  dockerA1            trace.to_MG01.row_count              2  2021-03-07 11:04:00 CST  2021-03-07 11:29:00 CST     4.000000      1.000    65.950000     5.800000,                  timestamp   cmdb_id                                            message
0  2021-03-07 11:29:04 CST  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
1  2021-03-07 11:29:09 CST  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing df_error if available; otherwise load
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')

# Ensure timestamp numeric
df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce').astype('Int64')

# Incident window (inclusive)
start_ts = 1615086000
end_ts = 1615087800
mask_window = (df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts)

# Store filtered error rows in kernel variable as requested (full rows retained)
error_logs_window = df_error.loc[mask_window].copy()

# Prepare compact display table: convert timestamp to UTC+8 readable and truncate message to 200 chars
if not error_logs_window.empty:
    error_logs_window_display = error_logs_window.loc[:, ['timestamp','cmdb_id','message']].copy()
    error_logs_window_display['timestamp'] = error_logs_window_display['timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    error_logs_window_display['message'] = error_logs_window_display['message'].astype(str).str.slice(0,200)
    # keep compact to at most 20 rows
    error_logs_window_display = error_logs_window_display.head(20)
else:
    error_logs_window_display = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Aggregate count per cmdb_id within the window, sorted by count desc (show top 20)
error_counts = error_logs_window.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False)
error_counts_top = error_counts.head(20)

# Display compact outputs (variables)
error_logs_window_display, error_counts_top

```
Out[6]:
```
```python
# Build a plain-English summary string of the filtered error logs (using existing kernel variables)
try:
    error_logs_window
    error_logs_window_display
    error_counts_top
except NameError:
    # Fallback: load and filter directly if variables missing
    import pandas as pd, pytz
    tz = pytz.timezone('Asia/Shanghai')
    df_error = pd.read_csv('error_logs.csv')
    df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce').astype('Int64')
    start_ts = 1615086000; end_ts = 1615087800
    error_logs_window = df_error.loc[(df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts)].copy()
    error_logs_window_display = error_logs_window.loc[:, ['timestamp','cmdb_id','message']].copy()
    error_logs_window_display['timestamp'] = error_logs_window_display['timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    error_logs_window_display['message'] = error_logs_window_display['message'].astype(str).str.slice(0,200)
    error_counts_top = error_logs_window.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

total_rows = int(error_logs_window.shape[0]) if not error_logs_window.empty else 0
top_counts = [(row.cmdb_id, int(row.count)) for row in error_counts_top.itertuples(index=False)]
# Prepare sample rows (up to 5)
sample_rows = []
for r in error_logs_window_display.head(5).itertuples(index=False):
    ts, cmdb, msg = r
    sample_rows.append(f"{ts} | {cmdb} | {msg}")

summary_lines = [
    "Error logs in window 1615086000–1615087800 (UTC+8):",
    f" - total rows: {total_rows}",
    f" - counts by cmdb_id (top): {', '.join(f'{c} ({n})' for c,n in top_counts) if top_counts else 'none'}",
    " - sample rows (timestamp UTC+8 | cmdb_id | message snippet):"
]
if sample_rows:
    for s in sample_rows:
        summary_lines.append(f"    - {s}")
else:
    summary_lines.append("    - (no rows)")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 timestamp   cmdb_id                                            message
0  2021-03-07 11:29:04 CST  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
1  2021-03-07 11:29:09 CST  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...,     cmdb_id  count
0  apache02      2)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timestamp and value types
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Define window (inclusive)
start_ts = 1615087720
end_ts = 1615087800

# Filter and store full filtered rows in required kernel variable
mask_window = (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)
trace_window_11_28_11_30 = df_trace.loc[mask_window].copy()

# Prepare display table: convert timestamp to UTC+8 readable string, select columns, sort
trace_window_11_28_11_30_display = trace_window_11_28_11_30.loc[:, ['timestamp','cmdb_id','trace_name','value']].copy()
if not trace_window_11_28_11_30_display.empty:
    trace_window_11_28_11_30_display['timestamp'] = trace_window_11_28_11_30_display['timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    trace_window_11_28_11_30_display = trace_window_11_28_11_30_display.sort_values(['cmdb_id','timestamp'], ascending=[True,True]).head(50)
else:
    trace_window_11_28_11_30_display = trace_window_11_28_11_30_display.head(0)

# Aggregate counts per cmdb_id within the window, sorted desc and limited to top 20
trace_window_counts = trace_window_11_28_11_30.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# Return the display DataFrame and the counts DataFrame (and keep full filtered rows stored in trace_window_11_28_11_30)
trace_window_11_28_11_30_display, trace_window_counts

```
Out[7]:
```
```python
# Summarize the filtered trace window results as a concise plain-English string.
try:
    trace_window_11_28_11_30_display
    trace_window_counts
except NameError:
    # Fallback: reload and filter if variables missing
    import pandas as pd, pytz
    tz = pytz.timezone('Asia/Shanghai')
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
    start_ts = 1615087720; end_ts = 1615087800
    mask = (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)
    trace_window_11_28_11_30 = df_trace.loc[mask].copy()
    trace_window_11_28_11_30_display = trace_window_11_28_11_30.loc[:, ['timestamp','cmdb_id','trace_name','value']].copy()
    trace_window_11_28_11_30_display['timestamp'] = trace_window_11_28_11_30_display['timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    trace_window_11_28_11_30_display = trace_window_11_28_11_30_display.sort_values(['cmdb_id','timestamp']).head(50)
    trace_window_counts = trace_window_11_28_11_30.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# Build summary
top_counts_list = [f"{row.cmdb_id} ({int(row.count)})" for row in trace_window_counts.itertuples(index=False)]
# capture up to 3 example rows
examples = []
for r in trace_window_11_28_11_30_display.head(3).itertuples(index=False):
    examples.append(f"{r.cmdb_id} | {r.trace_name} = {r.value}")

summary_lines = [
    "Trace window summary for timestamps 1615087720–1615087800 (~2021-03-07 11:28–11:30 CST):",
    f" - Sample returned: up to 50 trace rows (sorted by cmdb_id then timestamp).",
    f" - Top components by trace-row count in this window (top 20 shown): {', '.join(top_counts_list)}",
    " - Example trace rows (cmdb_id | trace_name = value):"
] + [f"    - {e}" for e in examples] + [
    " - Full filtered rows available in variable: trace_window_11_28_11_30",
    " - Counts per component available in variable: trace_window_counts"
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    timestamp cmdb_id                         trace_name       value
6645  2021-03-07 11:29:00 CST    IG01           trace.self.duration_mean   46.764648
6646  2021-03-07 11:29:00 CST    IG01            trace.self.duration_p95   60.000000
6647  2021-03-07 11:29:00 CST    IG01               trace.self.row_count  432.000000
6648  2021-03-07 11:29:00 CST    IG01    trace.to_Tomcat01.duration_mean  217.599217
6649  2021-03-07 11:29:00 CST    IG01     trace.to_Tomcat01.duration_p95  326.909650
...                       ...     ...                                ...         ...
6690  2021-03-07 11:29:00 CST    MG01  trace.from_dockerA2.duration_mean    0.006167
6691  2021-03-07 11:29:00 CST    MG01   trace.from_dockerA2.duration_p95    0.008500
6692  2021-03-07 11:29:00 CST    MG01      trace.from_dockerA2.row_count    6.000000
6693  2021-03-07 11:29:00 CST    MG01  trace.from_dockerB1.duration_mean    0.068625
6694  2021-03-07 11:29:00 CST    MG01   trace.from_dockerB1.duration_p95    0.259500

[50 rows x 4 columns],      cmdb_id  count
3       MG02     39
2       MG01     39
1       IG02     15
0       IG01     15
4   Tomcat01     15
5   Tomcat02     15
6   Tomcat03     15
7   Tomcat04     15
8   dockerA1     15
9   dockerA2     15
10  dockerB1     15
11  dockerB2     15)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure proper dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Ensure global thresholds exist (computed on full series BEFORE filtering) per rule 9
try:
    metric_kpi_thresholds
except NameError:
    metric_kpi_thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        count='count',
        mean='mean',
        std='std',
        p95=lambda x: x.quantile(0.95),
        p90=lambda x: x.quantile(0.90),
        p50=lambda x: x.quantile(0.50),
        p10=lambda x: x.quantile(0.10),
        p5=lambda x: x.quantile(0.05)
    ).reset_index()
    metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']] = metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']].round(6)
    metric_kpi_thresholds['count'] = metric_kpi_thresholds['count'].astype(int)

# Define window and candidate cmdb_ids
start_ts = 1615087720
end_ts = 1615087800
candidates = ['apache02','IG01','IG02','Tomcat01','Tomcat02','Tomcat03','Tomcat04','MG01','MG02','Redis02']

# Filter metric rows to the window and candidates, store full filtered rows in kernel variable
mask_window = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
mask_candidate = df_metric['cmdb_id'].isin(candidates)
metric_window_11_28_11_30 = df_metric.loc[mask_window & mask_candidate].copy()

# Join with global p95 from metric_kpi_thresholds
p95_lookup = metric_kpi_thresholds.loc[:, ['cmdb_id','kpi_name','p95']]
metric_window_11_28_11_30 = metric_window_11_28_11_30.merge(p95_lookup, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# Mark anomalies where value > p95 (strict), only when both numeric
metric_window_11_28_11_30['is_anomaly'] = False
mask_valid = metric_window_11_28_11_30['value'].notna() & metric_window_11_28_11_30['p95'].notna()
metric_window_11_28_11_30.loc[mask_valid, 'is_anomaly'] = metric_window_11_28_11_30.loc[mask_valid, 'value'] > metric_window_11_28_11_30.loc[mask_valid, 'p95']

# Aggregate anomalies by cmdb_id and kpi_name
anom_rows = metric_window_11_28_11_30.loc[metric_window_11_28_11_30['is_anomaly']].copy()

if anom_rows.shape[0] == 0:
    # No anomalies found: produce an empty summary frame with required columns
    metric_window_anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value','p95','severity_ratio'
    ])
else:
    agg = anom_rows.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count = ('value','count'),
        earliest_anomaly_ts = ('timestamp','min'),
        latest_anomaly_ts = ('timestamp','max'),
        max_value = ('value','max'),
        p95 = ('p95','first')
    ).reset_index()
    # Compute severity_ratio safely (avoid division by zero)
    agg['severity_ratio'] = np.where(agg['p95'].astype(float) != 0,
                                    (agg['max_value'] - agg['p95']) / agg['p95'],
                                    np.nan)
    # Convert earliest/latest to UTC+8 readable strings
    agg['earliest_anomaly_ts'] = agg['earliest_anomaly_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    agg['latest_anomaly_ts'] = agg['latest_anomaly_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    # Round numeric columns for compactness
    agg['max_value'] = agg['max_value'].round(6)
    agg['p95'] = agg['p95'].round(6)
    agg['severity_ratio'] = agg['severity_ratio'].round(6)
    # Sort by severity_ratio desc and keep top 20
    metric_window_anomaly_summary = agg.sort_values('severity_ratio', ascending=False).loc[
        :, ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value','p95','severity_ratio']
    ].head(20)

# Final compact output: metric_window_11_28_11_30 (full filtered rows stored) and metric_window_anomaly_summary (compact)
metric_window_11_28_11_30.shape, metric_window_anomaly_summary

```
Out[8]:
```
```python
# Plain-English summary of the metric-window anomaly results (compact)
summary = (
    "Metric window summary (1615087720–1615087800 ~ 2021-03-07 11:28–11:30 CST):\n\n"
    "- Total metric rows in the filtered window for selected components: 952.\n\n"
    "- Top anomalous component-KPI groups by severity_ratio (highest first):\n"
    "  1) Tomcat03 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT: 1 anomaly at 2021-03-07 11:30 CST, "
    "max=59.0 vs p95=2.95 (severity_ratio ≈ 19.0) — very large relative spike in CLOSE-WAIT.\n"
    "  2) apache02 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT: 2 anomalies (11:29–11:30 CST), "
    "max=3.0 vs p95=0.5 (severity_ratio = 5.0) — notable relative rise in CLOSE-WAIT.\n"
    "  3) Tomcat03 | Tomcat Threads (http-nio-8003 CurrentThreads): 2 anomalies (11:29–11:30 CST), "
    "max=142 vs p95=35 (severity_ratio ≈ 3.06) — thread usage >> normal.\n"
    "  4) Tomcat04 | Tomcat Threads (http-nio-8003 CurrentThreads): 2 anomalies, "
    "max=167 vs p95=48 (severity_ratio ≈ 2.48) — high thread usage.\n"
    "  5) Tomcat01/Tomcat02 also show elevated thread counts (severity_ratio >2 for some groups).\n"
    "  6) IG01 | JVM HeapMemoryUsage: 1 anomaly at 11:29 CST, max≈43.45 vs p95≈35.92 (severity_ratio ≈0.21) — memory higher than usual.\n\n"
    "- Observations & recommendations:\n"
    "  * The most urgent signals are network TCP CLOSE-WAIT spikes (Tomcat03 and apache02) and large increases in Tomcat connector threads across multiple Tomcat instances around 11:29–11:30. These could indicate connection handling issues or thread exhaustion impacting service throughput.\n"
    "  * IG01 shows a single-point elevated JVM heap usage — monitor for persistence or growth (possible OOM risk if sustained).\n"
    "  * Most other anomalies are single-minute spikes (anomaly_count typically 1 or 2), so investigate correlated traces/logs around 11:29–11:30 (error logs and trace spikes were observed in that window).\n\n"
    "Priority action: investigate Tomcat thread usage and TCP CLOSE-WAIT conditions first, then review IG01 JVM memory if elevation persists."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((952, 6),      cmdb_id                                           kpi_name  anomaly_count      earliest_anomaly_ts        latest_anomaly_ts  max_value        p95  severity_ratio
64  Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST    59.0000    2.95000       19.000000
94  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT              2  2021-03-07 11:29:00 CST  2021-03-07 11:30:00 CST     3.0000    0.50000        5.000000
70  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              2  2021-03-07 11:29:00 CST  2021-03-07 11:30:00 CST   142.0000   35.00000        3.057143
86  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              2  2021-03-07 11:29:00 CST  2021-03-07 11:30:00 CST   167.0000   48.00000        2.479167
49  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              2  2021-03-07 11:29:00 CST  2021-03-07 11:30:00 CST   162.0000   48.50000        2.340206
62  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              2  2021-03-07 11:29:00 CST  2021-03-07 11:30:00 CST   121.0000   37.00000        2.270270
85  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST   168.0000  100.00000        0.680000
48  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST   163.0000  100.00000        0.630000
77  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps              1  2021-03-07 11:29:00 CST  2021-03-07 11:29:00 CST    11.0000    7.00000        0.571429
69  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST   143.0000  100.00000        0.430000
9       IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST     0.3000    0.22500        0.333333
61  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST   122.0000  100.00000        0.220000
0       IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage              1  2021-03-07 11:29:00 CST  2021-03-07 11:29:00 CST    43.4472   35.92490        0.209390
92  apache02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST    10.4000    9.46669        0.098589
82  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST    50.0000   48.00000        0.041667
95  apache02    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST   420.0000  404.00000        0.039604
91  apache02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST     0.6833    0.65999        0.035319
76  Tomcat04   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite              1  2021-03-07 11:29:00 CST  2021-03-07 11:29:00 CST    11.0667   10.70670        0.033624
75  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps              1  2021-03-07 11:29:00 CST  2021-03-07 11:29:00 CST     0.9333    0.90333        0.033177
29      MG01                        OSLinux-CPU_CPU_CPUidleutil              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST    74.6529   72.34930        0.031840)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames if available; otherwise load from CSV
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')

# Ensure types
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')
df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce').astype('Int64')

# Window bounds
start_ts = 1615087720
end_ts = 1615087800

# 1) Trace rows where trace_name mentions Tomcat03 (including to_/from_) within the window
mask_trace_window = (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)
# match 'tomcat03' or 'to_tomcat03' or 'from_tomcat03' (case-insensitive). 'tomcat03' alone covers to_/from_, but keep regex explicit.
trace_name_regex = r'(to[_\-]?tomcat03|from[_\-]?tomcat03|tomcat03)'
trace_tomcat03_window = df_trace.loc[mask_trace_window & df_trace['trace_name'].str.contains(trace_name_regex, case=False, na=False)].copy()

# Prepare display: convert timestamp to UTC+8 readable, select columns, sort by timestamp ascending, limit 50
trace_tomcat03_display = trace_tomcat03_window.loc[:, ['timestamp','cmdb_id','trace_name','value']].copy()
trace_tomcat03_display['timestamp'] = trace_tomcat03_display['timestamp'].apply(
    lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
)
trace_tomcat03_display = trace_tomcat03_display.sort_values('timestamp', ascending=True).head(50)

# 2) Metric rows for cmdb_id == 'Tomcat03' within the window and filtered kpi_name patterns
mask_metric_window = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
mask_tomcat03 = df_metric['cmdb_id'] == 'Tomcat03'
kpi_regex = r'(CLOSE-WAIT|Tomcat-Threads|CurrentThreads|ErrorCount)'
metric_tomcat03_window = df_metric.loc[mask_metric_window & mask_tomcat03 & df_metric['kpi_name'].str.contains(kpi_regex, case=False, na=False)].copy()

# Convert timestamp to UTC+8 readable for display
metric_tomcat03_display = metric_tomcat03_window.loc[:, ['timestamp','cmdb_id','kpi_name','value']].copy()
metric_tomcat03_display['timestamp'] = metric_tomcat03_display['timestamp'].apply(
    lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
)
metric_tomcat03_display = metric_tomcat03_display.sort_values('timestamp', ascending=True).head(20)

# 2b) Metric anomalies for Tomcat03 in that window using precomputed global p95 (metric_kpi_thresholds)
# Ensure metric_kpi_thresholds exists (computed on full series before filtering)
try:
    metric_kpi_thresholds
except NameError:
    # compute global thresholds if missing (must be computed on full series)
    metric_kpi_thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        count='count',
        mean='mean',
        std='std',
        p95=lambda x: x.quantile(0.95),
        p90=lambda x: x.quantile(0.90),
        p50=lambda x: x.quantile(0.50),
        p10=lambda x: x.quantile(0.10),
        p5=lambda x: x.quantile(0.05)
    ).reset_index()
    metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']] = metric_kpi_thresholds[['mean','std','p95','p90','p50','p10','p5']].round(6)
    metric_kpi_thresholds['count'] = metric_kpi_thresholds['count'].astype(int)

# Join p95 for Tomcat03 metric rows
p95_lookup = metric_kpi_thresholds.loc[:, ['cmdb_id','kpi_name','p95']]
metric_tomcat03_window = metric_tomcat03_window.merge(p95_lookup, on=['cmdb_id','kpi_name'], how='left', validate='m:1')
# Flag anomalies where value > p95 (strict), only when both numeric
metric_tomcat03_window['is_anomaly'] = False
mask_valid_metric = metric_tomcat03_window['value'].notna() & metric_tomcat03_window['p95'].notna()
metric_tomcat03_window.loc[mask_valid_metric, 'is_anomaly'] = metric_tomcat03_window.loc[mask_valid_metric, 'value'] > metric_tomcat03_window.loc[mask_valid_metric, 'p95']

# Aggregate metric anomalies counts for Tomcat03 per kpi_name (if any)
if metric_tomcat03_window['is_anomaly'].any():
    tomcat03_metric_anomaly_counts = metric_tomcat03_window.loc[metric_tomcat03_window['is_anomaly']].groupby('kpi_name').agg(
        anomaly_count=('value','count'),
        earliest_anomaly_ts=('timestamp','min'),
        latest_anomaly_ts=('timestamp','max'),
        max_value=('value','max'),
        p95=('p95','first')
    ).reset_index()
    # Convert earliest/latest to readable UTC+8 strings and round numerics
    tomcat03_metric_anomaly_counts['earliest_anomaly_ts'] = tomcat03_metric_anomaly_counts['earliest_anomaly_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    tomcat03_metric_anomaly_counts['latest_anomaly_ts'] = tomcat03_metric_anomaly_counts['latest_anomaly_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    tomcat03_metric_anomaly_counts['max_value'] = tomcat03_metric_anomaly_counts['max_value'].round(6)
    tomcat03_metric_anomaly_counts['p95'] = tomcat03_metric_anomaly_counts['p95'].round(6)
    tomcat03_metric_anomaly_counts = tomcat03_metric_anomaly_counts.sort_values('anomaly_count', ascending=False).reset_index(drop=True)
else:
    tomcat03_metric_anomaly_counts = pd.DataFrame(columns=['kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value','p95'])

# 3) Logs: from log.csv and error_logs.csv for cmdb_id in ['Tomcat03','apache02'] within window
candidates_logs = ['Tomcat03','apache02']
mask_logs_window = (df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts) & df_log['cmdb_id'].isin(candidates_logs)
logs_filtered = df_log.loc[mask_logs_window].copy()
logs_display = logs_filtered.loc[:, ['timestamp','cmdb_id','log_name']].copy()
if not logs_display.empty:
    logs_display['timestamp'] = logs_display['timestamp'].apply(lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z'))
    logs_display = logs_display.sort_values(['cmdb_id','timestamp']).head(20)
else:
    logs_display = logs_display.head(0)

# Error logs filtered
mask_error_window = (df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts) & df_error['cmdb_id'].isin(candidates_logs)
error_logs_filtered = df_error.loc[mask_error_window].copy()
error_logs_display = error_logs_filtered.loc[:, ['timestamp','cmdb_id','message']].copy()
if not error_logs_display.empty:
    error_logs_display['timestamp'] = error_logs_display['timestamp'].apply(lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z'))
    error_logs_display['message'] = error_logs_display['message'].astype(str).str.slice(0,200)
    error_logs_display = error_logs_display.sort_values(['cmdb_id','timestamp']).head(20)
else:
    error_logs_display = error_logs_display.head(0)

# Aggregates:
# Counts of returned trace rows per cmdb_id (for the requested trace filters)
trace_counts_per_cmdb = trace_tomcat03_window.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)

# metric anomalies count for Tomcat03 already computed in tomcat03_metric_anomaly_counts (or empty DF)

# Keep variables in kernel as requested:
# - trace_tomcat03_window (full filtered trace rows for Tomcat03-related traces)
# - metric_tomcat03_window (filtered metric rows for Tomcat03 with p95 and is_anomaly)
# - logs_display, error_logs_display, trace_counts_per_cmdb, tomcat03_metric_anomaly_counts

trace_tomcat03_window, trace_tomcat03_display, trace_counts_per_cmdb, metric_tomcat03_window, metric_tomcat03_display, tomcat03_metric_anomaly_counts, logs_display, error_logs_display

```
Out[9]:
```
```python
# Summarize Tomcat03 involvement in the narrow window (1615087720–1615087800) as plain English.
try:
    trace_tomcat03_window
    trace_tomcat03_display
    trace_counts_per_cmdb
    metric_tomcat03_window
    metric_tomcat03_display
    tomcat03_metric_anomaly_counts
    logs_display
    error_logs_display
except NameError:
    raise RuntimeError("Required intermediate variables not found in kernel. Please run the previous analysis cells first.")

# Compute simple aggregates
trace_rows_total = int(trace_tomcat03_window.shape[0])
trace_counts = trace_counts_per_cmdb.copy()
trace_counts_list = ", ".join(f"{r.cmdb_id} ({int(r.count)})" for r in trace_counts.itertuples(index=False)) if not trace_counts.empty else "none"

metric_rows_total = int(metric_tomcat03_window.shape[0])
metric_anomaly_total = int(tomcat03_metric_anomaly_counts['anomaly_count'].sum()) if not tomcat03_metric_anomaly_counts.empty else 0

# Gather top metric anomaly summaries (if any)
top_metric_anoms = []
if not tomcat03_metric_anomaly_counts.empty:
    for r in tomcat03_metric_anomaly_counts.itertuples(index=False):
        kpi = r.kpi_name
        cnt = int(r.anomaly_count)
        start = r.earliest_anomaly_ts
        end = r.latest_anomaly_ts
        maxv = float(r.max_value)
        p95 = float(r.p95)
        top_metric_anoms.append(f"{kpi} | count={cnt} | earliest={start} | latest={end} | max={maxv} | p95={p95}")
top_metric_anoms_text = "\n    - ".join(top_metric_anoms) if top_metric_anoms else "None"

# Example trace rows (up to 3) showing heavy to/from Tomcat03 durations
example_traces = []
for r in trace_tomcat03_display.head(3).itertuples(index=False):
    example_traces.append(f"{r.timestamp} | {r.cmdb_id} | {r.trace_name} = {r.value}")
example_traces_text = "\n    - ".join(example_traces) if example_traces else "None"

# Logs summary
logs_count = int(logs_display.shape[0]) if 'logs_display' in globals() else 0
err_logs_count = int(error_logs_display.shape[0]) if 'error_logs_display' in globals() else 0

# Build summary string
summary_lines = [
    "Tomcat03 involvement summary (window 1615087720–1615087800 ≈ 2021-03-07 11:28–11:30 CST):",
    "",
    f"1) Trace evidence:",
    f" - Trace rows matching Tomcat03 (to_/from_/mention) in window: {trace_rows_total}",
    f" - Counts by source component: {trace_counts_list}",
    " - Example trace samples (timestamp UTC+8 | cmdb_id | trace_name = value):",
    f"    - {example_traces_text}",
    "",
    "Interpretation: IG01 and IG02 show large 'to_Tomcat03' duration_mean values at 11:29 (~224–254) with substantial row_count (e.g., 48, 41). MG01/MG02 show 'from_Tomcat03' traces with smaller row_counts.",
    "",
    "2) Metric evidence for Tomcat03:",
    f" - Total Tomcat03 metric rows in window (filtered KPIs): {metric_rows_total}",
    f" - Total metric anomaly points (value > global p95) for Tomcat03: {metric_anomaly_total}",
    " - Top metric anomalies (kpi | count | earliest | latest | max | p95):",
    f"    - {top_metric_anoms_text}",
    "",
    "Key metric highlights: multiple Tomcat-Threads CurrentThreads metrics exceed their p95 at 11:29–11:30 (e.g., max 142 vs p95 35; max 143 vs p95 100), and a TCP CLOSE-WAIT spike (Tomcat03 max=59 vs p95≈2.95) at 11:30.",
    "",
    "3) Logs in window for Tomcat03/apache02:",
    f" - log.csv rows returned: {logs_count} (see sample variable 'logs_display')",
    f" - error_logs.csv rows returned: {err_logs_count} (see sample variable 'error_logs_display')",
    "",
    "Conclusion / next steps:",
    " - Strong evidence implicates Tomcat03 around 11:29–11:30: high request durations reaching Tomcat03 from IG01/IG02, spikes in Tomcat connector thread counts, and a TCP CLOSE-WAIT spike. Prioritize investigating Tomcat03 thread exhaustion, connection handling (CLOSE-WAIT), and related request traces/logs.",
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       timestamp cmdb_id                         trace_name       value
6654  1615087740    IG01    trace.to_Tomcat03.duration_mean  224.475063
6655  1615087740    IG01     trace.to_Tomcat03.duration_p95  327.251150
6656  1615087740    IG01        trace.to_Tomcat03.row_count   48.000000
6669  1615087740    IG02    trace.to_Tomcat03.duration_mean  253.586610
6670  1615087740    IG02     trace.to_Tomcat03.duration_p95  328.806000
6671  1615087740    IG02        trace.to_Tomcat03.row_count   41.000000
6681  1615087740    MG01  trace.from_Tomcat03.duration_mean    3.103615
6682  1615087740    MG01   trace.from_Tomcat03.duration_p95   15.070400
6683  1615087740    MG01      trace.from_Tomcat03.row_count   13.000000
6720  1615087740    MG02  trace.from_Tomcat03.duration_mean    0.247500
6721  1615087740    MG02   trace.from_Tomcat03.duration_p95    1.106450
6722  1615087740    MG02      trace.from_Tomcat03.row_count   10.000000,                     timestamp cmdb_id                         trace_name       value
6654  2021-03-07 11:29:00 CST    IG01    trace.to_Tomcat03.duration_mean  224.475063
6655  2021-03-07 11:29:00 CST    IG01     trace.to_Tomcat03.duration_p95  327.251150
6656  2021-03-07 11:29:00 CST    IG01        trace.to_Tomcat03.row_count   48.000000
6669  2021-03-07 11:29:00 CST    IG02    trace.to_Tomcat03.duration_mean  253.586610
6670  2021-03-07 11:29:00 CST    IG02     trace.to_Tomcat03.duration_p95  328.806000
6671  2021-03-07 11:29:00 CST    IG02        trace.to_Tomcat03.row_count   41.000000
6681  2021-03-07 11:29:00 CST    MG01  trace.from_Tomcat03.duration_mean    3.103615
6682  2021-03-07 11:29:00 CST    MG01   trace.from_Tomcat03.duration_p95   15.070400
6683  2021-03-07 11:29:00 CST    MG01      trace.from_Tomcat03.row_count   13.000000
6720  2021-03-07 11:29:00 CST    MG02  trace.from_Tomcat03.duration_mean    0.247500
6721  2021-03-07 11:29:00 CST    MG02   trace.from_Tomcat03.duration_p95    1.106450
6722  2021-03-07 11:29:00 CST    MG02      trace.from_Tomcat03.row_count   10.000000,   cmdb_id  count
0    IG01      3
1    IG02      3
2    MG01      3
3    MG02      3,     timestamp   cmdb_id                                           kpi_name  value     p95  is_anomaly
0  1615087740  Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0    2.95       False
1  1615087740  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...   30.0   30.00       False
2  1615087740  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  100.0  100.00       False
3  1615087740  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...   60.0   35.00        True
4  1615087740  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  500.0  500.00       False
5  1615087800  Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   59.0    2.95        True
6  1615087800  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...   30.0   30.00       False
7  1615087800  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  143.0  100.00        True
8  1615087800  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  142.0   35.00        True
9  1615087800  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  500.0  500.00       False,                      timestamp   cmdb_id                                           kpi_name  value
26952  2021-03-07 11:29:00 CST  Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0
26963  2021-03-07 11:29:00 CST  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...   30.0
26979  2021-03-07 11:29:00 CST  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  100.0
26980  2021-03-07 11:29:00 CST  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...   60.0
26981  2021-03-07 11:29:00 CST  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  500.0
27496  2021-03-07 11:30:00 CST  Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   59.0
27507  2021-03-07 11:30:00 CST  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...   30.0
27523  2021-03-07 11:30:00 CST  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  143.0
27524  2021-03-07 11:30:00 CST  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  142.0
27525  2021-03-07 11:30:00 CST  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  500.0,                                             kpi_name  anomaly_count      earliest_anomaly_ts        latest_anomaly_ts  max_value     p95
0  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              2  2021-03-07 11:29:00 CST  2021-03-07 11:30:00 CST      142.0   35.00
1     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST       59.0    2.95
2  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              1  2021-03-07 11:30:00 CST  2021-03-07 11:30:00 CST      143.0  100.00,                    timestamp   cmdb_id         log_name
492  2021-03-07 11:29:00 CST  Tomcat03  log.error_count
493  2021-03-07 11:29:00 CST  Tomcat03    log.row_count
498  2021-03-07 11:29:00 CST  apache02  log.error_count
499  2021-03-07 11:29:00 CST  apache02    log.row_count
500  2021-03-07 11:30:00 CST  apache02  log.error_count
501  2021-03-07 11:30:00 CST  apache02    log.row_count,                  timestamp   cmdb_id                                            message
0  2021-03-07 11:29:04 CST  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
1  2021-03-07 11:29:09 CST  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...)```
```